In [58]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
import random
import numpy as np
import copy
import matplotlib.pyplot as plt


import os
os.environ["CUDA_VISIBLE_DEVICES"]="1"

In [59]:
# Set device (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [60]:
import torch
import torch.nn as nn
import torch.optim as optim

import torch
import torch.nn as nn
import torch.nn.functional as F

class MLP(nn.Module):
    def __init__(self, input_size=784, hidden_size=256, num_classes=10, n_layers=4):
        super(MLP, self).__init__()
        self.n_layers = n_layers

        # First layer weights and biases
        self.fc1_weight = nn.Parameter(torch.randn(hidden_size, input_size) * 0.01)
        self.fc1_bias = nn.Parameter(torch.zeros(hidden_size))

        # Hidden layers
        self.fcs_weight = nn.ParameterList([
            nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01)
            for _ in range(n_layers)
        ])
        self.fcs_bias = nn.ParameterList([
            nn.Parameter(torch.zeros(hidden_size))
            for _ in range(n_layers)
        ])

        # Output layer
        self.fc2_weight = nn.Parameter(torch.randn(num_classes, hidden_size) * 0.01)
        self.fc2_bias = nn.Parameter(torch.zeros(num_classes))

        self.relu = nn.ReLU()

        # SNIP masks (used only during pruning)
        self.c1_weight = torch.ones_like(self.fc1_weight, requires_grad=False).to(device)
        self.c1_bias = torch.ones_like(self.fc1_bias, requires_grad=False).to(device)
        self.cs_weight = [torch.ones_like(w, requires_grad=False).to(device) for w in self.fcs_weight]
        self.cs_bias = [torch.ones_like(b, requires_grad=False).to(device) for b in self.fcs_bias]
        self.c2_weight = torch.ones_like(self.fc2_weight, requires_grad=False).to(device)
        self.c2_bias = torch.ones_like(self.fc2_bias, requires_grad=False).to(device)

    def forward(self, x):
        x = x.view(-1, 784)

        x = F.linear(x, self.fc1_weight * self.c1_weight, self.fc1_bias * self.c1_bias)
        x = self.relu(x)

        for i in range(self.n_layers):
            x = F.linear(x, self.fcs_weight[i] * self.cs_weight[i], self.fcs_bias[i] * self.cs_bias[i])
            x = self.relu(x)

        x = F.linear(x, self.fc2_weight * self.c2_weight, self.fc2_bias * self.c2_bias)
        return x

    def snip(self, train_loader, sparsity_level=0.5):
        # Zero previous gradients
        self.zero_grad()

        # Enable gradients for masks
        self.c1_weight.requires_grad_(True)
        for c in self.cs_weight:
            c.requires_grad_(True)
        self.c2_weight.requires_grad_(True)
        self.c1_bias.requires_grad_(True)
        for c in self.cs_bias:
            c.requires_grad_(True)
        self.c2_bias.requires_grad_(True)

        loss = 0
        for i, (images, labels) in enumerate(train_loader):
            if i >= 10:
                break
            images, labels = images.to(device), labels.to(device)

            # Forward pass
            outputs = self.forward(images)
            loss += F.cross_entropy(outputs, labels)

        loss.backward()

        # Collect all saliency scores |∂L/∂c|
        grads = [torch.abs(self.c1_weight.grad)]
        grads += [torch.abs(self.c1_bias.grad)]
        for j in range(self.n_layers):
            grads += [torch.abs(self.cs_weight[j].grad), torch.abs(self.cs_bias[j])]
        grads += [torch.abs(self.c2_weight.grad)]
        grads += [torch.abs(self.c2_bias.grad)]

        # Concatenate all scores
        all_scores = torch.cat([g.flatten() for g in grads])
        num_params_to_keep = int((1 - sparsity_level) * all_scores.numel())

        # Determine threshold
        threshold, _ = torch.kthvalue(all_scores, all_scores.numel() - num_params_to_keep)
        masks = [g >= threshold for g in grads]

        # Apply binary masks (disconnect grads from original)
        with torch.no_grad():
            i = 0
            self.c1_weight.data = masks[i].float().to(device)
            i += 1
            self.c1_bias.data = masks[i].float().to(device)
            i += 1
            for j in range(self.n_layers):
                self.cs_weight[j].data = masks[i].float().to(device)
                i += 1
                self.cs_bias[j].data = masks[i].float().to(device)
                i += 1

            self.c2_weight.data = masks[i].float().to(device)
            i += 1
            self.c2_bias.data = masks[i].float().to(device)
            i += 1

        # Detach masks to avoid keeping computation graph
        self.c1_weight.requires_grad_(False)
        for c in self.cs_weight:
            c.requires_grad_(False)
        self.c2_weight.requires_grad_(False)
        self.c1_bias.requires_grad_(False)
        for c in self.cs_bias:
            c.requires_grad_(False)
        self.c2_bias.requires_grad_(False)

        return

class MorphologicalLayer(nn.Module):
    def __init__(self, input_size, output_size, bias=True, alpha=1, beta=0):
        super(MorphologicalLayer, self).__init__()
        self.bias = bias

        self.w = nn.Parameter(torch.randn(output_size, input_size)*alpha - beta)
        if bias:
            self.b = nn.Parameter(torch.randn(output_size))
            self.b2 = nn.Parameter(torch.randn(output_size))
            self.c_bias = torch.ones_like(self.b, requires_grad=False).to(device)
            self.c2_bias = torch.ones_like(self.b2, requires_grad=False).to(device)

        self.c_weight = torch.ones_like(self.w).to(device)
        self.c_weight.requires_grad = False

        self.mode = "snip"

    def forward(self, x, dropout=None):
        x = x.unsqueeze(1) 
        if dropout:
            mask = torch.rand(self.w.size(0), self.w.size(1)) < dropout
            mask = mask.to(device)
        max_w = self.w
        min_w = self.w
        if dropout:
            max_w = max_w - mask * 1e9
            min_w = min_w + mask * 1e9
        if self.mode == "normal":
            max_w = max_w - (1-self.c_weight) * 1e9
            min_w = min_w + (1-self.c_weight) * 1e9
        else:
            max_w = max_w * self.c_weight
            min_w = min_w * self.c_weight
        x_max = x + max_w
        x_min = x + min_w
        if self.bias:
            if self.mode == "normal":
                max_b = self.b - (1-self.c_bias) * 1e9
                min_b = self.b2 + (1-self.c2_bias) * 1e9
            else:
                max_b = self.b * self.c_bias
                min_b = self.b2 * self.c2_bias
            x_max = torch.cat([x_max, max_b.view(1,-1,1).repeat(x.size(0),1,1)], dim=2)
            x_min = torch.cat([x_min, min_b.view(1,-1,1).repeat(x.size(0),1,1)], dim=2)
        x_max = torch.clamp(x_max, min=-1e8)
        x_min = torch.clamp(x_min, max=1e8)
        x_max, _ = torch.max(x_max, dim=2)
        x_min, _ = torch.min(x_min, dim=2)
        x = x_max + x_min
        return x
    
class LinAct(nn.Module):
    def __init__(self, size, method="simple"):
        super(LinAct, self).__init__()
        self.size = size
        self.method = method
        if method == "simple":
            self.a = nn.Parameter(torch.randn(size) / 3.46)
            # self.a = nn.Parameter(torch.rand(size)-0.5)
            self.c = torch.ones_like(self.a, requires_grad=False).to(device)
        else:
            tmp = nn.Linear(size, size, bias = False).weight.detach()
            U, S, Vh = np.linalg.svd(tmp.cpu().numpy(), full_matrices=True)
            self.U = torch.tensor(U).to(device)
            self.V = torch.tensor(Vh).t().to(device)
            self.a = torch.zeros(size).to(device)
            self.a[:S.shape[0]] = torch.tensor(S)
            self.a = nn.Parameter(self.a)

    def forward(self, x):
        if self.method == "simple":
            masked_a = self.a * self.c
            x = x * masked_a.view(1, -1).repeat(x.size(0), 1)
        else:
            Smat = torch.diag(self.a)
            tmp = self.U @ Smat @ self.V.t()
            x = torch.mm(x, tmp.t())
        return x
    
class MPM(nn.Module):
    def __init__(self, input_size=784, hidden_size=256, num_classes=10, n_layers=4):
        super(MPM, self).__init__()
        self.morph_layer1 = MorphologicalLayer(input_size, hidden_size, alpha=0)
        self.linact1 = LinAct(hidden_size)
        self.morph_layers = nn.ModuleList([nn.Sequential(MorphologicalLayer(hidden_size, hidden_size, alpha=1), LinAct(hidden_size)) for _ in range(n_layers)])
        self.morph_layer2 = MorphologicalLayer(hidden_size, num_classes, alpha=1)

    def forward(self, x):
        x = x.view(-1, 784)  # Flatten input
        x = self.morph_layer1(x)
        x = self.linact1(x)
        for layer in self.morph_layers:
            x = layer(x)
        x = self.morph_layer2(x)
        return x
    
    def snip(self, train_loader, sparsity_level=0.5):
        # Zero previous gradients
        self.zero_grad()

        # Enable gradients for masks
        self.morph_layer1.c_weight.requires_grad_(True)
        if self.morph_layer1.bias:
            self.morph_layer1.c_bias.requires_grad_(True)
            self.morph_layer1.c2_bias.requires_grad_(True)
        self.linact1.c.requires_grad_(True)
        for layer in self.morph_layers:
            layer[0].c_weight.requires_grad_(True)
            if layer[0].bias:
                layer[0].c_bias.requires_grad_(True)
                layer[0].c2_bias.requires_grad_(True)
            layer[1].c.requires_grad_(True)
        self.morph_layer2.c_weight.requires_grad_(True)
        if self.morph_layer2.bias:
            self.morph_layer2.c_bias.requires_grad_(True)
            self.morph_layer2.c2_bias.requires_grad_(True)

        self.morph_layer1.mode = "snip"
        for layer in self.morph_layers:
            layer[0].mode = "snip"
        self.morph_layer2.mode = "snip"

        loss = 0
        for i, (images, labels) in enumerate(train_loader):
            if i >= 10:
                break
            images, labels = images.to(device), labels.to(device)

            # Forward pass
            outputs = self.forward(images)
            loss += F.cross_entropy(outputs, labels)

        loss.backward()

        # Collect all saliency scores |∂L/∂c|
        grads = [torch.abs(self.morph_layer1.c_weight.grad)]
        if self.morph_layer1.bias:
            grads += [torch.abs(self.morph_layer1.c_bias.grad)]
            grads += [torch.abs(self.morph_layer1.c2_bias.grad)]
        grads += [torch.abs(self.linact1.c.grad)]
        for layer in self.morph_layers:
            grads += [torch.abs(layer[0].c_weight.grad)] 
            if layer[0].bias:
                grads += [torch.abs(layer[0].c_bias.grad)]
                grads += [torch.abs(layer[0].c2_bias.grad)]
            grads += [torch.abs(layer[1].c.grad)]
        grads += [torch.abs(self.morph_layer2.c_weight.grad)]
        if self.morph_layer2.bias:
            grads += [torch.abs(self.morph_layer2.c_bias.grad)]
            grads += [torch.abs(self.morph_layer2.c2_bias.grad)]        

        # Concatenate all scores
        all_scores = torch.cat([g.flatten() for g in grads])
        num_params_to_keep = int((1 - sparsity_level) * all_scores.numel())

        # Determine threshold
        threshold, _ = torch.kthvalue(all_scores, all_scores.numel() - num_params_to_keep)
        masks = [g >= threshold for g in grads]

        # Apply binary masks (disconnect grads from original)
        with torch.no_grad():
            i = 0
            self.morph_layer1.c_weight.data = masks[i].float().to(device)
            i += 1
            if self.morph_layer1.bias:
                self.morph_layer1.c_bias.data = masks[i].float().to(device)
                i += 1
                self.morph_layer1.c2_bias.data = masks[i].float().to(device)
                i += 1
            self.linact1.c.data = masks[i].float().to(device)
            i += 1
            for layer in self.morph_layers:
                layer[0].c_weight.data = masks[i].float().to(device)
                i += 1
                if layer[0].bias:
                    layer[0].c_bias.data = masks[i].float().to(device)
                    i += 1
                    layer[0].c2_bias.data = masks[i].float().to(device)
                    i += 1
                layer[1].c.data = masks[i].float().to(device)
                i += 1

            self.morph_layer2.c_weight.data = masks[i].float().to(device)
            i += 1
            if self.morph_layer2.bias:
                self.morph_layer2.c_bias.data = masks[i].float().to(device)
                i += 1
                self.morph_layer2.c2_bias.data = masks[i].float().to(device)
                i += 1

        # Detach masks to avoid keeping computation graph
        self.morph_layer1.c_weight.requires_grad_(False)
        if self.morph_layer1.bias:
            self.morph_layer1.c_bias.requires_grad_(False)
            self.morph_layer1.c2_bias.requires_grad_(False)
        self.linact1.c.requires_grad_(False)
        for layer in self.morph_layers:
            layer[0].c_weight.requires_grad_(False)
            if layer[0].bias:
                layer[0].c_bias.requires_grad_(False)
                layer[0].c2_bias.requires_grad_(False)
            layer[1].c.requires_grad_(False)
        self.morph_layer2.c_weight.requires_grad_(False)
        if self.morph_layer2.bias:
            self.morph_layer2.c_bias.requires_grad_(False)
            self.morph_layer2.c2_bias.requires_grad_(False)

        self.morph_layer1.mode = "normal"
        for layer in self.morph_layers:
            layer[0].mode = "normal"
        self.morph_layer2.mode = "normal"

        return

In [100]:
def train(model, criterion, optimizer, train_loader, val_loader, num_epochs=50, return_list=False):
    # Training and validation loop
    best_val_accuracy = 0.0
    best_model = None

    train_list = []
    val_list = []

    for epoch in range(num_epochs):
        # Training phase
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # Validation phase
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in train_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        train_accuracy = 100 * correct / total
        train_list.append(train_accuracy)
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_accuracy = 100 * correct / total
        val_list.append(val_accuracy)
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Train Accuracy: {train_accuracy:.2f}%, Validation Accuracy: {val_accuracy:.2f}%")

        # Save best model based on validation accuracy
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_model = copy.deepcopy(model)

    if return_list:
        return best_model, train_list, val_list
    else:
        return best_model

def test(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f'Accuracy on the test set: {accuracy:.2f}%')

In [62]:
# Load and preprocess MNIST dataset
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

full_train_dataset = torchvision.datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, transform=transform, download=True)

# Split train dataset into training and validation sets
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [67]:
# Initialize model, loss function, and optimizer
model = MLP().to(device)

model.snip(train_loader, sparsity_level=0.9925)

total_params = 0
for param in model.parameters():
    total_params += param.numel() * (1-0.9925)
print(f"Total number of parameters: {int(total_params)}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model = train(model, criterion, optimizer, train_loader, val_loader)

Total number of parameters: 3500
Epoch [1/50], Loss: 1.8901, Train Accuracy: 33.39%, Validation Accuracy: 32.47%
Epoch [2/50], Loss: 1.5878, Train Accuracy: 43.84%, Validation Accuracy: 43.37%
Epoch [3/50], Loss: 1.5148, Train Accuracy: 48.98%, Validation Accuracy: 48.27%
Epoch [4/50], Loss: 1.1419, Train Accuracy: 55.42%, Validation Accuracy: 54.61%
Epoch [5/50], Loss: 1.3372, Train Accuracy: 58.84%, Validation Accuracy: 57.74%
Epoch [6/50], Loss: 1.2866, Train Accuracy: 60.62%, Validation Accuracy: 59.76%
Epoch [7/50], Loss: 1.3407, Train Accuracy: 62.01%, Validation Accuracy: 61.01%
Epoch [8/50], Loss: 1.0183, Train Accuracy: 62.31%, Validation Accuracy: 61.67%
Epoch [9/50], Loss: 1.4837, Train Accuracy: 62.75%, Validation Accuracy: 62.01%
Epoch [10/50], Loss: 1.1650, Train Accuracy: 63.09%, Validation Accuracy: 62.19%
Epoch [11/50], Loss: 1.2933, Train Accuracy: 63.41%, Validation Accuracy: 62.38%
Epoch [12/50], Loss: 1.2198, Train Accuracy: 63.65%, Validation Accuracy: 63.05%
Epoc

In [68]:
test(model, test_loader)

Accuracy on the test set: 71.45%


In [71]:
# Initialize model, loss function, and optimizer
model = MPM().to(device)

model.snip(train_loader, sparsity_level=0.9925)

total_params = 0
for param in model.parameters():
    total_params += param.numel() * (1-0.9925)
print(f"Total number of parameters: {int(total_params)}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model = train(model, criterion, optimizer, train_loader, val_loader)

Total number of parameters: 3519
Epoch [1/50], Loss: 0.7157, Train Accuracy: 83.54%, Validation Accuracy: 83.23%
Epoch [2/50], Loss: 0.3998, Train Accuracy: 90.89%, Validation Accuracy: 90.23%
Epoch [3/50], Loss: 0.1417, Train Accuracy: 92.89%, Validation Accuracy: 91.72%
Epoch [4/50], Loss: 0.1669, Train Accuracy: 94.01%, Validation Accuracy: 92.72%
Epoch [5/50], Loss: 0.2517, Train Accuracy: 94.78%, Validation Accuracy: 93.29%
Epoch [6/50], Loss: 0.2219, Train Accuracy: 95.36%, Validation Accuracy: 93.51%
Epoch [7/50], Loss: 0.2496, Train Accuracy: 95.82%, Validation Accuracy: 93.61%
Epoch [8/50], Loss: 0.1951, Train Accuracy: 96.30%, Validation Accuracy: 93.85%
Epoch [9/50], Loss: 0.2090, Train Accuracy: 96.69%, Validation Accuracy: 93.96%
Epoch [10/50], Loss: 0.2154, Train Accuracy: 96.92%, Validation Accuracy: 93.97%
Epoch [11/50], Loss: 0.1022, Train Accuracy: 97.31%, Validation Accuracy: 94.41%
Epoch [12/50], Loss: 0.1184, Train Accuracy: 97.57%, Validation Accuracy: 94.42%
Epoc

In [72]:
test(model, test_loader)

Accuracy on the test set: 94.73%


In [73]:
# Initialize model, loss function, and optimizer
model = MLP().to(device)

model.snip(train_loader, sparsity_level=0.995)

total_params = 0
for param in model.parameters():
    total_params += param.numel() * (1-0.995)
print(f"Total number of parameters: {int(total_params)}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model = train(model, criterion, optimizer, train_loader, val_loader)

Total number of parameters: 2333
Epoch [1/50], Loss: 2.3009, Train Accuracy: 11.27%, Validation Accuracy: 11.12%
Epoch [2/50], Loss: 2.2950, Train Accuracy: 11.27%, Validation Accuracy: 11.12%
Epoch [3/50], Loss: 2.2979, Train Accuracy: 11.27%, Validation Accuracy: 11.12%
Epoch [4/50], Loss: 2.2999, Train Accuracy: 11.27%, Validation Accuracy: 11.12%
Epoch [5/50], Loss: 2.3097, Train Accuracy: 11.27%, Validation Accuracy: 11.12%
Epoch [6/50], Loss: 2.2913, Train Accuracy: 11.27%, Validation Accuracy: 11.12%
Epoch [7/50], Loss: 2.3074, Train Accuracy: 11.27%, Validation Accuracy: 11.12%
Epoch [8/50], Loss: 2.3079, Train Accuracy: 11.27%, Validation Accuracy: 11.12%
Epoch [9/50], Loss: 2.3016, Train Accuracy: 11.27%, Validation Accuracy: 11.12%
Epoch [10/50], Loss: 2.3114, Train Accuracy: 11.27%, Validation Accuracy: 11.12%
Epoch [11/50], Loss: 2.3042, Train Accuracy: 11.27%, Validation Accuracy: 11.12%
Epoch [12/50], Loss: 2.3050, Train Accuracy: 11.27%, Validation Accuracy: 11.12%
Epoc

In [74]:
test(model, test_loader)

Accuracy on the test set: 11.35%


In [75]:
# Initialize model, loss function, and optimizer
model = MPM().to(device)

model.snip(train_loader, sparsity_level=0.995)

total_params = 0
for param in model.parameters():
    total_params += param.numel() * (1-0.995)
print(f"Total number of parameters: {int(total_params)}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model = train(model, criterion, optimizer, train_loader, val_loader)

Total number of parameters: 2346
Epoch [1/50], Loss: 0.7708, Train Accuracy: 83.62%, Validation Accuracy: 83.70%
Epoch [2/50], Loss: 0.3021, Train Accuracy: 89.37%, Validation Accuracy: 88.67%
Epoch [3/50], Loss: 0.3668, Train Accuracy: 91.80%, Validation Accuracy: 90.86%
Epoch [4/50], Loss: 0.1796, Train Accuracy: 93.08%, Validation Accuracy: 92.12%
Epoch [5/50], Loss: 0.1818, Train Accuracy: 93.90%, Validation Accuracy: 92.60%
Epoch [6/50], Loss: 0.2592, Train Accuracy: 94.46%, Validation Accuracy: 92.86%
Epoch [7/50], Loss: 0.3026, Train Accuracy: 95.41%, Validation Accuracy: 93.34%
Epoch [8/50], Loss: 0.1815, Train Accuracy: 95.79%, Validation Accuracy: 93.53%
Epoch [9/50], Loss: 0.3791, Train Accuracy: 96.26%, Validation Accuracy: 93.83%
Epoch [10/50], Loss: 0.1994, Train Accuracy: 96.57%, Validation Accuracy: 93.72%
Epoch [11/50], Loss: 0.0862, Train Accuracy: 96.79%, Validation Accuracy: 93.73%
Epoch [12/50], Loss: 0.0417, Train Accuracy: 97.11%, Validation Accuracy: 93.99%
Epoc

In [76]:
test(model, test_loader)

Accuracy on the test set: 94.16%


In [77]:
# Initialize model, loss function, and optimizer
model = MLP().to(device)

model.snip(train_loader, sparsity_level=0.9975)

total_params = 0
for param in model.parameters():
    total_params += param.numel() * (1-0.9975)
print(f"Total number of parameters: {int(total_params)}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model = train(model, criterion, optimizer, train_loader, val_loader)

Total number of parameters: 1166


Epoch [1/50], Loss: 2.3026, Train Accuracy: 9.77%, Validation Accuracy: 10.29%
Epoch [2/50], Loss: 2.3026, Train Accuracy: 9.77%, Validation Accuracy: 10.29%
Epoch [3/50], Loss: 2.3026, Train Accuracy: 9.77%, Validation Accuracy: 10.29%
Epoch [4/50], Loss: 2.3026, Train Accuracy: 9.77%, Validation Accuracy: 10.29%
Epoch [5/50], Loss: 2.3026, Train Accuracy: 9.77%, Validation Accuracy: 10.29%
Epoch [6/50], Loss: 2.3026, Train Accuracy: 9.77%, Validation Accuracy: 10.29%
Epoch [7/50], Loss: 2.3026, Train Accuracy: 9.77%, Validation Accuracy: 10.29%
Epoch [8/50], Loss: 2.3026, Train Accuracy: 9.77%, Validation Accuracy: 10.29%
Epoch [9/50], Loss: 2.3026, Train Accuracy: 9.77%, Validation Accuracy: 10.29%
Epoch [10/50], Loss: 2.3026, Train Accuracy: 9.77%, Validation Accuracy: 10.29%
Epoch [11/50], Loss: 2.3026, Train Accuracy: 9.77%, Validation Accuracy: 10.29%
Epoch [12/50], Loss: 2.3026, Train Accuracy: 9.77%, Validation Accuracy: 10.29%
Epoch [13/50], Loss: 2.3026, Train Accuracy: 9.77

In [78]:
test(model, test_loader)

Accuracy on the test set: 9.80%


In [79]:
# Initialize model, loss function, and optimizer
model = MPM().to(device)

model.snip(train_loader, sparsity_level=0.9975)

total_params = 0
for param in model.parameters():
    total_params += param.numel() * (1-0.9975)
print(f"Total number of parameters: {int(total_params)}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model = train(model, criterion, optimizer, train_loader, val_loader)

Total number of parameters: 1173
Epoch [1/50], Loss: 0.7309, Train Accuracy: 85.33%, Validation Accuracy: 85.11%
Epoch [2/50], Loss: 0.4289, Train Accuracy: 90.22%, Validation Accuracy: 89.69%
Epoch [3/50], Loss: 0.2596, Train Accuracy: 92.23%, Validation Accuracy: 91.56%
Epoch [4/50], Loss: 0.3762, Train Accuracy: 93.50%, Validation Accuracy: 92.44%
Epoch [5/50], Loss: 0.1442, Train Accuracy: 94.31%, Validation Accuracy: 92.87%
Epoch [6/50], Loss: 0.1621, Train Accuracy: 94.94%, Validation Accuracy: 93.27%
Epoch [7/50], Loss: 0.2010, Train Accuracy: 95.65%, Validation Accuracy: 93.71%
Epoch [8/50], Loss: 0.2590, Train Accuracy: 96.05%, Validation Accuracy: 93.80%
Epoch [9/50], Loss: 0.2153, Train Accuracy: 96.29%, Validation Accuracy: 93.89%
Epoch [10/50], Loss: 0.0938, Train Accuracy: 96.73%, Validation Accuracy: 94.30%
Epoch [11/50], Loss: 0.1681, Train Accuracy: 97.13%, Validation Accuracy: 94.23%
Epoch [12/50], Loss: 0.0706, Train Accuracy: 97.32%, Validation Accuracy: 94.47%
Epoc

In [80]:
test(model, test_loader)

Accuracy on the test set: 94.57%


In [101]:
# Load and preprocess MNIST dataset
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

full_train_dataset = torchvision.datasets.FashionMNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = torchvision.datasets.FashionMNIST(root='./data', train=False, transform=transform, download=True)

# Split train dataset into training and validation sets
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [85]:
# Initialize model, loss function, and optimizer
model = MLP().to(device)

model.snip(train_loader, sparsity_level=0.9925)

total_params = 0
for param in model.parameters():
    total_params += param.numel() * (1-0.9925)
print(f"Total number of parameters: {int(total_params)}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model = train(model, criterion, optimizer, train_loader, val_loader)

Total number of parameters: 3500
Epoch [1/50], Loss: 2.1363, Train Accuracy: 19.60%, Validation Accuracy: 19.52%
Epoch [2/50], Loss: 1.8186, Train Accuracy: 21.61%, Validation Accuracy: 21.52%
Epoch [3/50], Loss: 1.9948, Train Accuracy: 21.53%, Validation Accuracy: 21.23%
Epoch [4/50], Loss: 1.9607, Train Accuracy: 21.31%, Validation Accuracy: 21.13%
Epoch [5/50], Loss: 1.8532, Train Accuracy: 21.27%, Validation Accuracy: 21.07%
Epoch [6/50], Loss: 2.0061, Train Accuracy: 21.26%, Validation Accuracy: 21.07%
Epoch [7/50], Loss: 1.9687, Train Accuracy: 20.99%, Validation Accuracy: 20.68%
Epoch [8/50], Loss: 1.9414, Train Accuracy: 20.85%, Validation Accuracy: 20.51%
Epoch [9/50], Loss: 1.8802, Train Accuracy: 21.02%, Validation Accuracy: 20.52%
Epoch [10/50], Loss: 1.8580, Train Accuracy: 21.30%, Validation Accuracy: 20.67%
Epoch [11/50], Loss: 2.1019, Train Accuracy: 21.19%, Validation Accuracy: 20.63%
Epoch [12/50], Loss: 1.9284, Train Accuracy: 21.36%, Validation Accuracy: 20.73%
Epoc

In [86]:
test(model, test_loader)

Accuracy on the test set: 21.99%


In [102]:
# Initialize model, loss function, and optimizer
model = MPM().to(device)

model.snip(train_loader, sparsity_level=0.9925)

total_params = 0
for param in model.parameters():
    total_params += param.numel() * (1-0.9925)
print(f"Total number of parameters: {int(total_params)}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model = train(model, criterion, optimizer, train_loader, val_loader)

Total number of parameters: 3519
Epoch [1/50], Loss: 1.0063, Train Accuracy: 72.86%, Validation Accuracy: 72.40%
Epoch [2/50], Loss: 0.5910, Train Accuracy: 80.22%, Validation Accuracy: 78.92%
Epoch [3/50], Loss: 0.5275, Train Accuracy: 83.49%, Validation Accuracy: 81.29%
Epoch [4/50], Loss: 0.4578, Train Accuracy: 85.54%, Validation Accuracy: 82.19%
Epoch [5/50], Loss: 0.3062, Train Accuracy: 86.69%, Validation Accuracy: 82.62%
Epoch [6/50], Loss: 0.4881, Train Accuracy: 87.96%, Validation Accuracy: 83.12%
Epoch [7/50], Loss: 0.3636, Train Accuracy: 88.61%, Validation Accuracy: 83.33%
Epoch [8/50], Loss: 0.3000, Train Accuracy: 89.40%, Validation Accuracy: 83.33%
Epoch [9/50], Loss: 0.3670, Train Accuracy: 89.82%, Validation Accuracy: 83.22%
Epoch [10/50], Loss: 0.3737, Train Accuracy: 90.71%, Validation Accuracy: 83.63%
Epoch [11/50], Loss: 0.3589, Train Accuracy: 91.15%, Validation Accuracy: 83.47%
Epoch [12/50], Loss: 0.2226, Train Accuracy: 91.66%, Validation Accuracy: 83.38%
Epoc

In [103]:
test(model, test_loader)

Accuracy on the test set: 83.12%


In [89]:
# Initialize model, loss function, and optimizer
model = MLP().to(device)

model.snip(train_loader, sparsity_level=0.995)

total_params = 0
for param in model.parameters():
    total_params += param.numel() * (1-0.995)
print(f"Total number of parameters: {int(total_params)}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model = train(model, criterion, optimizer, train_loader, val_loader)

Total number of parameters: 2333
Epoch [1/50], Loss: 2.3026, Train Accuracy: 10.10%, Validation Accuracy: 9.58%
Epoch [2/50], Loss: 2.3025, Train Accuracy: 10.10%, Validation Accuracy: 9.58%
Epoch [3/50], Loss: 2.3026, Train Accuracy: 10.10%, Validation Accuracy: 9.58%
Epoch [4/50], Loss: 2.3025, Train Accuracy: 10.10%, Validation Accuracy: 9.58%
Epoch [5/50], Loss: 2.3025, Train Accuracy: 10.10%, Validation Accuracy: 9.58%
Epoch [6/50], Loss: 2.3027, Train Accuracy: 10.10%, Validation Accuracy: 9.58%
Epoch [7/50], Loss: 2.3024, Train Accuracy: 10.10%, Validation Accuracy: 9.58%
Epoch [8/50], Loss: 2.3025, Train Accuracy: 10.10%, Validation Accuracy: 9.58%
Epoch [9/50], Loss: 2.3026, Train Accuracy: 10.10%, Validation Accuracy: 9.58%
Epoch [10/50], Loss: 2.3028, Train Accuracy: 10.10%, Validation Accuracy: 9.58%
Epoch [11/50], Loss: 2.3028, Train Accuracy: 10.10%, Validation Accuracy: 9.58%
Epoch [12/50], Loss: 2.3024, Train Accuracy: 10.10%, Validation Accuracy: 9.58%
Epoch [13/50], L

In [90]:
test(model, test_loader)

Accuracy on the test set: 10.00%


In [104]:
# Initialize model, loss function, and optimizer
model = MPM().to(device)

model.snip(train_loader, sparsity_level=0.995)

total_params = 0
for param in model.parameters():
    total_params += param.numel() * (1-0.995)
print(f"Total number of parameters: {int(total_params)}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model = train(model, criterion, optimizer, train_loader, val_loader)

Total number of parameters: 2346
Epoch [1/50], Loss: 0.9582, Train Accuracy: 74.21%, Validation Accuracy: 73.78%
Epoch [2/50], Loss: 0.5762, Train Accuracy: 81.03%, Validation Accuracy: 79.76%
Epoch [3/50], Loss: 0.6563, Train Accuracy: 83.76%, Validation Accuracy: 81.37%
Epoch [4/50], Loss: 0.4336, Train Accuracy: 85.45%, Validation Accuracy: 82.31%
Epoch [5/50], Loss: 0.3321, Train Accuracy: 86.66%, Validation Accuracy: 83.20%
Epoch [6/50], Loss: 0.2787, Train Accuracy: 87.71%, Validation Accuracy: 82.99%
Epoch [7/50], Loss: 0.3803, Train Accuracy: 88.55%, Validation Accuracy: 83.31%
Epoch [8/50], Loss: 0.2429, Train Accuracy: 89.41%, Validation Accuracy: 83.13%
Epoch [9/50], Loss: 0.3475, Train Accuracy: 89.80%, Validation Accuracy: 82.91%
Epoch [10/50], Loss: 0.2964, Train Accuracy: 90.75%, Validation Accuracy: 83.28%
Epoch [11/50], Loss: 0.2530, Train Accuracy: 91.17%, Validation Accuracy: 83.41%
Epoch [12/50], Loss: 0.1998, Train Accuracy: 91.51%, Validation Accuracy: 83.57%
Epoc

In [105]:
test(model, test_loader)

Accuracy on the test set: 82.53%


In [93]:
# Initialize model, loss function, and optimizer
model = MLP().to(device)

model.snip(train_loader, sparsity_level=0.9975)

total_params = 0
for param in model.parameters():
    total_params += param.numel() * (1-0.9975)
print(f"Total number of parameters: {int(total_params)}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model = train(model, criterion, optimizer, train_loader, val_loader)

Total number of parameters: 1166


Epoch [1/50], Loss: 2.3026, Train Accuracy: 10.00%, Validation Accuracy: 10.01%
Epoch [2/50], Loss: 2.3026, Train Accuracy: 10.00%, Validation Accuracy: 10.01%
Epoch [3/50], Loss: 2.3026, Train Accuracy: 10.00%, Validation Accuracy: 10.01%
Epoch [4/50], Loss: 2.3026, Train Accuracy: 10.00%, Validation Accuracy: 10.01%
Epoch [5/50], Loss: 2.3026, Train Accuracy: 10.00%, Validation Accuracy: 10.01%
Epoch [6/50], Loss: 2.3026, Train Accuracy: 10.00%, Validation Accuracy: 10.01%
Epoch [7/50], Loss: 2.3026, Train Accuracy: 10.00%, Validation Accuracy: 10.01%
Epoch [8/50], Loss: 2.3026, Train Accuracy: 10.00%, Validation Accuracy: 10.01%
Epoch [9/50], Loss: 2.3026, Train Accuracy: 10.00%, Validation Accuracy: 10.01%
Epoch [10/50], Loss: 2.3026, Train Accuracy: 10.00%, Validation Accuracy: 10.01%
Epoch [11/50], Loss: 2.3026, Train Accuracy: 10.00%, Validation Accuracy: 10.01%
Epoch [12/50], Loss: 2.3026, Train Accuracy: 10.00%, Validation Accuracy: 10.01%
Epoch [13/50], Loss: 2.3026, Train Ac

In [94]:
test(model, test_loader)

Accuracy on the test set: 10.00%


In [106]:
# Initialize model, loss function, and optimizer
model = MPM().to(device)

model.snip(train_loader, sparsity_level=0.9975)

total_params = 0
for param in model.parameters():
    total_params += param.numel() * (1-0.9975)
print(f"Total number of parameters: {int(total_params)}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model = train(model, criterion, optimizer, train_loader, val_loader)

Total number of parameters: 1173
Epoch [1/50], Loss: 0.8505, Train Accuracy: 73.67%, Validation Accuracy: 73.90%
Epoch [2/50], Loss: 0.6320, Train Accuracy: 81.41%, Validation Accuracy: 80.13%
Epoch [3/50], Loss: 0.3987, Train Accuracy: 84.35%, Validation Accuracy: 81.94%
Epoch [4/50], Loss: 0.4702, Train Accuracy: 86.10%, Validation Accuracy: 82.61%
Epoch [5/50], Loss: 0.5001, Train Accuracy: 87.48%, Validation Accuracy: 83.07%
Epoch [6/50], Loss: 0.5775, Train Accuracy: 88.30%, Validation Accuracy: 83.58%
Epoch [7/50], Loss: 0.5346, Train Accuracy: 89.65%, Validation Accuracy: 83.53%
Epoch [8/50], Loss: 0.2962, Train Accuracy: 90.16%, Validation Accuracy: 83.71%
Epoch [9/50], Loss: 0.3353, Train Accuracy: 90.99%, Validation Accuracy: 83.66%
Epoch [10/50], Loss: 0.2953, Train Accuracy: 91.60%, Validation Accuracy: 84.16%
Epoch [11/50], Loss: 0.2724, Train Accuracy: 92.05%, Validation Accuracy: 83.99%
Epoch [12/50], Loss: 0.3366, Train Accuracy: 92.61%, Validation Accuracy: 83.68%
Epoc

In [107]:
test(model, test_loader)

Accuracy on the test set: 82.95%
